# Building Your First AI Agent with LangChain

Welcome back.

In this lesson, we are going to build our first AI agent using the LangChain framework. You have already seen much of this code in the previous module, so we will move through the familiar concepts quickly and focus on understanding how everything fits together.

## 1. What Makes an Agent?

Let's start with the most important distinction.

A basic LLM call is usually a **one-shot interaction**:

**Input → LLM → Text Output → Done**

There is no tool usage, no dynamic decision-making, and no iterative reasoning.

An **AI agent** is fundamentally different.

An agent uses an LLM as a reasoning engine that can dynamically decide what to do next. It can:

* Call tools
* Inspect the results
* Decide whether it needs additional information
* Call another tool
* Repeat the process
* Eventually provide a final answer

This is the key difference between a simple LLM application and an agent.

A commonly used definition is:

> **An agent is a system where the LLM decides the control flow of the application.**

In a traditional application, the developer explicitly defines every step:

1. Do this.
2. Then do that.
3. Then call this API.
4. Then process the result.

With an agent, the LLM dynamically decides which tool to use and in what order.

This is what makes agents powerful and fundamentally different from simple chains.

---

# 2. Tools: Giving the LLM the Ability to Act

We introduced tools earlier in Module 1, so let's quickly recap them.

Tools are what transform a basic LLM from a system that only generates text into a system that can actually perform actions.

Without tools, an LLM can primarily generate responses based on the information it has learned.

With tools, an agent can:

* Search the web for current information
* Query databases
* Execute code
* Call APIs
* Perform calculations
* Read files
* Interact with external systems

In LangChain, tools are commonly defined as Python functions with:

* A name
* A description
* Type hints
* A return type

The LLM reads the tool's structured description and decides when and how to use it.

LangChain also provides many built-in tools, and you can create your own custom tools using the `@tool` decorator.

One important concept is that the LLM does **not directly execute your Python function**.

Instead, the process looks something like this:

**User Request → LLM → Tool Call Request → LangChain Runtime → Tool Execution → Result → LLM**

The LLM generates a structured request to call a tool. The LangChain runtime then executes the actual Python function.

This separation is extremely important because it makes the system more controlled, predictable, and safer.

---

# 3. Anatomy of a LangChain Tool

Let's look at the basic structure of a tool.

For example, imagine an addition function.

The first thing you might see is:

```python
@tool
```

This is called a **decorator**.

The decorator registers the Python function as a tool that the agent can discover and call.

Without the decorator, the agent does not automatically know that the function is available as a tool.

Think of the `@tool` decorator as putting a special label on a function that tells the agent framework:

> "This function can be used as a tool."

The next important concept is the **type hint**.

For example:

```python
def add(a: float, b: float) -> float:
```

The type hints tell the framework and the LLM what kind of inputs the function expects.

In this example:

* `a` expects a floating-point number.
* `b` expects a floating-point number.
* The function returns a floating-point number.

The next important component is the **docstring**.

For example:

```python
"""Add two numbers together. Use this for addition operations."""
```

The LLM reads this description to understand when the tool should be used.

The clearer your description is, the more accurately the agent can select the appropriate tool.

For example, if the user asks:

> "What is 25 plus 15?"

The LLM can look at the available tool descriptions and recognize that the `add` tool is appropriate.

---

# 4. From Python Function to JSON Schema

There is another important concept happening behind the scenes.

The LangChain framework takes your Python function and converts its structure into a format that the LLM can understand, typically a structured **JSON schema**.

The LLM does not simply receive your Python source code and execute it.

Instead, it receives a structured description containing information such as:

* Tool name
* Tool description
* Input parameters
* Input data types
* Output information

The LLM then uses this structured information to decide whether it should call the tool.

This is why **type hints and docstrings are so important** in agent development.

They are not only documentation for other developers.

They also provide important information that helps the AI understand:

* What the tool does
* When to use it
* What parameters it needs
* What kind of values to provide

So, when building tools for AI agents, your function names, descriptions, type hints, and return types all matter.

---

# 5. Understanding Python Decorators

Let's quickly clarify the Python concepts involved here.

A **decorator** is a Python feature that allows you to wrap or modify the behavior of a function.

In the context of LangChain, the `@tool` decorator essentially tells the framework:

> "Register this function as a callable tool that an agent can discover and use."

The underlying function remains a normal Python function, but the decorator gives it additional meaning within the agent framework.

You can think of `@tool` as a special label attached to the function.

---

# 6. Understanding Type Hints

Type hints describe what kind of data a function expects.

For example:

```python
def multiply(a: float, b: float) -> float:
```

This tells us:

* `a` should be a float.
* `b` should be a float.
* The function returns a float.

The agent framework can use this information to construct valid tool calls.

If the expected data type is not clearly defined, the model may have a harder time generating correct tool inputs.

Therefore, accurate type hints are an important part of reliable agent development.

---

# 7. Understanding Docstrings

A docstring is the triple-quoted description immediately inside a Python function.

For example:

```python
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
```

The docstring explains what the function does.

The LLM uses this description when deciding which tool to call.

A vague description can result in incorrect tool selection.

A clear description makes the tool's purpose obvious.

For example:

> "Multiply two numbers together."

is much better than:

> "Perform an operation."

The first description gives the model a clear indication of when the tool should be used.

---

# 8. Understanding Return Types

The return type tells the framework what kind of data the tool produces.

For example:

```python
-> float
```

means the function returns a floating-point number.

This information can help the agent understand how the output of one tool might be used as the input to another tool.

This becomes particularly important when an agent performs multiple steps.

For example:

1. Call `multiply`.
2. Receive the result.
3. Pass that result to `divide`.
4. Receive the final result.

The agent needs to understand how the outputs and inputs connect together.

Therefore, decorators, type hints, docstrings, and return types are all important when designing reliable tools.

---

# 9. The ReAct Pattern

We introduced the ReAct pattern earlier, so let's review it quickly.

**ReAct** stands for:

> **Reasoning + Acting**

It is one of the most common patterns used to build AI agents.

The agent follows a loop:

### Reason

The agent determines what it needs to do next.

### Act

The agent calls a tool or performs another action.

### Observe

The agent examines the result of that action.

Then it asks:

> "Do I have enough information to answer the user?"

If the answer is yes, it provides the final response.

If the answer is no, the agent reasons again, selects another action, observes the result, and continues.

The overall pattern looks like this:

**User Request → Reason → Act → Observe → Reason → Act → Observe → Final Answer**

This loop continues until the agent has enough information to complete the task.

---

# 10. Example: Weather Agent

Imagine the user asks:

> "What is the weather in Seattle?"

The agent may reason:

> "I need current weather information."

It then selects the appropriate weather tool.

The tool returns the current weather information.

The agent observes the result and determines:

> "I have enough information to answer the user."

It then provides the final answer.

For a more complex question, the agent might perform several tool calls.

For example:

1. Search for information.
2. Calculate a value.
3. Cross-check the result.
4. Search for additional information.
5. Provide the final answer.

This is the power of the ReAct pattern.

The agent is not following a rigid sequence that the developer manually programmed. Instead, it dynamically decides what to do next.

This is the fundamental pattern behind many LangChain-based agents.

---

# 11. Building Your First LangChain Agent

Now let's build a simple agent.

We will follow four basic steps.

### Step 1: Initialize the Model

First, we initialize the LLM.

For example, we can use an OpenAI chat model.

The model acts as the reasoning engine for the agent.

---

### Step 2: Define the Tools

Next, we define the tools that the agent can use.

To keep the example simple, we will create two mathematical tools:

* Multiply
* Divide

Each tool has:

* The `@tool` decorator
* A clear docstring
* Type hints
* A return type

The agent will use the descriptions of these tools to decide which one to call.

---

### Step 3: Create the Agent

This is the most important step.

Using LangChain's agent functionality, we provide:

* The model
* The available tools

The framework then creates the agent.

Conceptually, it looks like:

```python
agent = create_agent(
    model=model,
    tools=[multiply, divide]
)
```

This is the magic of the framework.

With a relatively small amount of code, LangChain and its underlying agent runtime handle the complex orchestration.

Behind the scenes, the agent runtime creates the reasoning loop.

The agent can:

1. Receive the user's question.
2. Reason about what needs to be done.
3. Select a tool.
4. Execute the tool.
5. Observe the result.
6. Decide whether another tool call is necessary.
7. Return the final answer.

---

# 12. Step 4: Invoke the Agent

Finally, we invoke the agent with a question.

For example:

> "What is 15 multiplied by 8 and then divided by 3?"

The agent has access to two tools:

* `multiply`
* `divide`

It must determine how to solve the problem.

The process looks like this:

### Loop 1

The agent recognizes that it needs to multiply first.

It calls:

```text
multiply(15, 8)
```

The result is:

```text
120
```

### Loop 2

The agent observes that the user still needs the result divided by 3.

It calls:

```text
divide(120, 3)
```

The result is:

```text
40
```

The agent now has enough information and returns:

> "The answer is 40."

The important point is that we did not explicitly program the sequence:

```text
Multiply first.
Then divide.
```

The agent figured out the sequence dynamically based on the user's request and the available tools.

This is what makes the system an **agent**.

---

# 13. Complete Agent Flow

The overall process can be summarized as:

```text
User:
What is 15 × 8 ÷ 3?

        ↓

LLM reasons:
I need to multiply first.

        ↓

Call Multiply Tool:
15 × 8

        ↓

Observation:
120

        ↓

LLM reasons:
Now I need to divide by 3.

        ↓

Call Divide Tool:
120 ÷ 3

        ↓

Observation:
40

        ↓

Final Answer:
40
```

The agent dynamically decides what to do at every step.

---

# 14. The Complete Agent in About 15 Lines

The basic structure is remarkably simple.

Conceptually, the application contains four parts:

```text
1. Initialize the model
2. Define the tools
3. Create the agent
4. Invoke the agent
```

The framework handles the complex reasoning and tool orchestration behind the scenes.

This means you can build a working agent with a surprisingly small amount of code.

The important thing is not the number of lines of code.

The important thing is understanding what happens behind those lines:

**LLM + Tools + ReAct Loop = Agent**

The LLM provides the reasoning capability.

The tools provide the ability to take actions.

The ReAct loop allows the agent to dynamically reason, act, observe, and continue until the task is complete.

---

# Key Takeaways

Let's summarize the most important concepts from this lesson.

### 1. An agent is different from a basic LLM call

A basic LLM call is typically one-shot.

An agent can dynamically decide what to do next.

### 2. The LLM controls the application flow

Instead of the developer explicitly defining every step, the LLM decides which tools to use and in what order.

### 3. Tools give agents the ability to act

Tools allow agents to interact with external systems, APIs, databases, code, and other services.

### 4. Tool descriptions matter

The LLM uses tool names, descriptions, type hints, and schemas to decide which tool to call.

### 5. The `@tool` decorator registers a function

It allows the LangChain framework to expose a Python function as a tool that an agent can discover and use.

### 6. Type hints and docstrings are important

They are not just for human developers. They also provide instructions and structured information that help the AI use tools correctly.

### 7. ReAct is the core agent pattern

The agent follows a cycle:

**Reason → Act → Observe → Repeat**

### 8. LangChain simplifies agent development

Instead of manually implementing the entire reasoning and tool execution loop, the framework provides abstractions that handle much of this complexity.

### 9. Agents dynamically plan their actions

In the multiplication and division example, the developer did not explicitly program the order of operations. The agent determined the sequence based on the user's request.

---


